<a href="https://colab.research.google.com/github/Ayu-sshhhhh/NLP-by-me/blob/main/Into_to_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to NLP Fundamentals
NLP has the goal of deriving info. from natural language (could be texts or sequence).
Another common term for NLP problems is sequence to sequence problems (seq2seq).

# Getting some pre-defined helper functions


In [ ]:
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py

from helper_functions import unzip_data, create_tensorboard_callback, plot_loss_curves, compare_historys

--2026-06-30 04:51:18--  https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10246 (10K) [text/plain]
Saving to: ‘helper_functions.py.1’

helper_functions.py 100%[===================>]  10.01K  --.-KB/s    in 0s      

2026-06-30 04:51:18 (77.0 MB/s) - ‘helper_functions.py.1’ saved [10246/10246]



# Get a text dataset
The dataset we're going to be using is Kaggle's introduction to NLP
See the original source here : https://www.kaggle.com/c/nlp-getting-started

In [ ]:
import pandas as pd
from pandas import read_csv
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

## Glimpse of Data

In [ ]:
train_df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [ ]:
train_df["text"][0]

'Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all'

In [ ]:
test_df.tail()

,id,keyword,location,text
3258,10861,NaN,NaN,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...
3259,10865,NaN,NaN,Storm in RI worse than last hurricane. My city...
3260,10868,NaN,NaN,Green Line derailment in Chicago http://t.co/U...
3261,10874,NaN,NaN,MEG issues Hazardous Weather Outlook (HWO) htt...
3262,10875,NaN,NaN,#CityofCalgary has activated its Municipal Eme...


In [ ]:
len(test_df), len(train_df)

(3263, 7613)

In [ ]:
# Shuffling the data (just for fun)
train_df_shuffled = train_df.sample(frac=1, random_state=42)
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


# Split data into train and validation sets

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled["text"].to_numpy(),
                                                                            train_df_shuffled["target"].to_numpy(),
                                                                            test_size=0.1,
                                                                            random_state=42)


In [ ]:
len(train_sentences)

6851

In [ ]:
len(val_sentences)

762

In [ ]:
train_sentences[:10], train_labels[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

# Convert texts to numbers

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

# This is a sample TextVectorizzation object
text_vectorizer = TextVectorization(max_tokens=None,
                                    standardize="lower_and_strip_punctuation",
                                    split="whitespace",
                                    ngrams=None,
                                    output_mode="int",
                                    output_sequence_length=None)

In [ ]:
max_vocab_length=10000
max_length = 15

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [20]:
text_vectorizer.adapt(train_sentences)

In [22]:
sample_sentence = "My name is Ana, I am from England. Nice to meet you !"
text_vectorizer([sample_sentence])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[  13,  735,    9,    1,    8,  160,   20, 2126, 1117,    5, 1389,
          12,    0,    0,    0]])>

### Tokenizing a random sentence

In [24]:
import random
random_sentence = random.choice(train_sentences)
print(f"Original Sentence : \n{random_sentence}\
\n\nVectorized version :")
text_vectorizer([random_sentence])

Original Sentence : 
Just stop fucking saying ÛÏa whole Û÷notherÛ. It just sounds fucking stupid. You fucking mean ÛÏa whole otherÛ. Not a fucking tongue-twister.

Vectorized version :


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[  29,  240,  255, 1179, 2588,  292, 6404,   15,   29,  919,  255,
        1755,   12,  255, 1274]])>

### Unique words in vocab

Here, [UNK] token is used for 'unknown' words

In [27]:
words_in_vocab = text_vectorizer.get_vocabulary()
top_5_words = words_in_vocab[:5]
bottom_5_words = words_in_vocab[-5:]
print(f"Number of words in vocab : {len(words_in_vocab)}")
print(f"Top 5 words in vocab : {top_5_words}")
print(f"Bottom 5 words in vocab : {bottom_5_words}")

Number of words in vocab : 10000
Top 5 words in vocab : ['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('in')]
Bottom 5 words in vocab : [np.str_('pages'), np.str_('paeds'), np.str_('pads'), np.str_('padres'), np.str_('paddytomlinson1')]


In [31]:
words_in_vocab[12:15]

[np.str_('you'), np.str_('my'), np.str_('with')]

# Cooking an Embedding layer

In [34]:
tf.random.set_seed(42)
from tensorflow.keras import layers
embedding = layers.Embedding(input_dim=max_vocab_length,
                             output_dim = 128,
                             embeddings_initializer="uniform",
                             name="embedding_1")
embedding

<Embedding name=embedding_1, built=False>

As seen above, `built=False` means :
Keras uses a *lazy initialization* strategy. When you define layers.Embedding(...), Keras registers the configuration you want (like the vocabulary size and output dimensions), but it does not actually create the weight matrices in memory yet.

It waits to officially "build" the layer and allocate memory until one of two things happens:

* You pass some actual data through the layer for the first time.
* You manually call the .build(input_shape) method.

In [35]:
sample_embed = embedding(text_vectorizer(["Hehe, haha, huhu"]))
sample_embed

<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[ 0.03804724, -0.01597831,  0.01647571, ..., -0.04200643,
          0.01338409,  0.0314011 ],
        [ 0.04713282,  0.00818918,  0.01831057, ...,  0.00969405,
         -0.02147803, -0.01891074],
        [ 0.03804724, -0.01597831,  0.01647571, ..., -0.04200643,
          0.01338409,  0.0314011 ],
        ...,
        [ 0.03224857,  0.01081032, -0.03953809, ..., -0.02717659,
          0.00401526, -0.00296935],
        [ 0.03224857,  0.01081032, -0.03953809, ..., -0.02717659,
          0.00401526, -0.00296935],
        [ 0.03224857,  0.01081032, -0.03953809, ..., -0.02717659,
          0.00401526, -0.00296935]]], dtype=float32)>

In [36]:
sample_embed[0][0]

<tf.Tensor: shape=(128,), dtype=float32, numpy=
array([ 0.03804724, -0.01597831,  0.01647571, -0.01125431, -0.03149595,
       -0.04279233,  0.01456931,  0.0263281 , -0.03230388,  0.02608326,
       -0.0329885 ,  0.0069576 ,  0.00617241, -0.00146865, -0.00622234,
       -0.00074804,  0.03253236, -0.01087886,  0.02498961, -0.03001909,
        0.04667917, -0.00766932, -0.03614656,  0.01526889, -0.03599138,
       -0.01936823, -0.01450326,  0.00163102, -0.04946984, -0.03327687,
        0.02723439,  0.04259627, -0.00362999, -0.02386597,  0.04752752,
        0.0281483 , -0.01283344,  0.02143333, -0.03463108, -0.00497477,
       -0.04366824, -0.02016079,  0.00119086, -0.01890732, -0.03081948,
        0.02160584, -0.03042245, -0.00839524,  0.04893697, -0.03526317,
       -0.04960076, -0.04772055, -0.04327276, -0.01410768, -0.03775169,
        0.01387539, -0.03458994,  0.04632438,  0.03809139,  0.0396078 ,
        0.00526441,  0.03224775,  0.01921458, -0.0360172 ,  0.04062624,
        0.006297